# Llava模型的部署

（使用的环境是 llava2）

In [2]:
from PIL import Image
import torch

from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path, tokenizer_image_token, process_images
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from transformers import TextStreamer

ModuleNotFoundError: No module named 'PIL'

In [1]:
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path
from llava.eval.run_llava import eval_model

model_path = "liuhaotian/llava-v1.5-7b"

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name=get_model_name_from_path(model_path)
)

/home/NCUT/teacher/xmy/anaconda3/envs/llava2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/NCUT/teacher/xmy/anaconda3/envs/llava2/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]/home/NCUT/teacher/xmy/anaconda3/envs/llava2/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedSto

In [4]:
device_id = 4  # 你想用的 GPU 编号，例如：cuda:2
device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")

In [9]:
model = model.to(device)  # 非常关键

# === 3. 加载图片 ===
image_path = "/home/NCUT/teacher/xmy/MRAG/PMI/test/url1.jpg"


# === 4. 设置 prompt ===
question = "Describe the scene in the image."
prompt = f"{DEFAULT_IMAGE_TOKEN} {question}"

# === 5. 图像 + 文本预处理 ===
# 加载图像
image = Image.open(image_path).convert("RGB")

# 用 image_processor 预处理
image_tensor = image_processor.preprocess(image, return_tensors="pt")["pixel_values"]  # shape: [1, 3, H, W]

# 移动到设备并转为 float16
image_tensor = image_tensor.to(device, dtype=torch.float16)


# === 6. 模型推理 ===
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)



In [16]:
input_ids

tensor([[    1,   529,  3027, 29958, 20355,   915,   278,  9088,   297,   278,
          1967, 29889]], device='cuda:4')

In [13]:
with torch.no_grad():
    output_ids = model.generate(
        input_ids=input_ids,
        images=image_tensor,
        do_sample=False,
        max_new_tokens=512,
        streamer=streamer
    )


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
# === 7. 解码输出 ===
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\nAnswer:", output_text)

### 下面是 整体的代码

In [3]:
import sys
sys.path.append("/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/LLaVA")

In [4]:
from PIL import Image
import torch

from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path, tokenizer_image_token, process_images
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from transformers import TextStreamer

model_path = "/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/LLaVA/llava-v1.5-7b"

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name=get_model_name_from_path(model_path)
)

device_id = 4  # 你想用的 GPU 编号，例如：cuda:2
device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
model = model.to(device)  # 非常关键


# 3. 图像预处理
image_path = "/home/NCUT/teacher/xmy/MRAG/PMI/test/url1.jpg"


images = [load_image_from_base64(image) for image in images]
                image_sizes = [image.size for image in images]
                images = process_images(images, image_processor, model.config)

# 4. 文本 token 化
question = "Describe the scene in the image."
prompt = f"USER: <image>\n{question} ASSISTANT:"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)


# === 5. 图像 + 文本预处理 ===
# 获得图片size
imgs = [image]  # 或多张
image_tensor = process_images(imgs, image_processor, model.config).to(device, dtype=torch.float16)
# 获取统一尺寸
_, _, H, W = image_tensor.shape
image_sizes = [(W, H)] * len(imgs)


# === 6. 模型推理 ===
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# 图像预处理
pixel_values = image_processor.preprocess(image, return_tensors="pt")["pixel_values"]


print("images.shape:", image_tensor.shape)
print("image_sizes:", image_sizes)  # 应输出：[(336, 336)]

with torch.no_grad():
    output_ids = model.generate(
        input_ids=input_ids,
        images=image_tensor,
        image_sizes=image_sizes,
        do_sample=False,
        max_new_tokens=512,
        streamer=streamer
    )


# === 7. 解码输出 ===
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\nAnswer:", output_text)

/home/NCUT/teacher/xmy/anaconda3/envs/Few_shot_llava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 2/2 [01:12<00:00, 36.02s/it]
You shouldn't move a model when it is dispatched on multiple devices.


images.shape: torch.Size([1, 3, 336, 336])
image_sizes: [(336, 336)]


AttributeError: 'NoneType' object has no attribute 'shape'

In [17]:
def make_square(img, size=672):
    w, h = img.size
    max_side = max(w, h)
    bg = Image.new('RGB', (max_side, max_side), (255,255,255))
    bg.paste(img, ((max_side-w)//2, (max_side-h)//2))
    return bg.resize((size,size), Image.BICUBIC)

orig = Image.open(image_path).convert("RGB")
square = make_square(orig, 672)
image_tensor = image_processor.preprocess(square, return_tensors="pt")["pixel_values"].to(device, dtype=torch.float16)
image_sizes = [(672, 672)]

prompt = f"USER: <image>\n{question} ASSISTANT:"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

print("images.shape:", image_tensor.shape)
print("image_sizes:", image_sizes)

images.shape: torch.Size([1, 3, 336, 336])
image_sizes: [(672, 672)]


In [ ]:
with torch.no_grad():
    output_ids = model.generate(
        input_ids=input_ids,
        images=image_tensor,
        image_sizes=image_sizes,
        do_sample=False,
        max_new_tokens=512,
        streamer=TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

### hf 版本

In [2]:
# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/llava-1.5-7b-hf")
model = AutoModelForImageTextToText.from_pretrained("/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/llava-1.5-7b-hf")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards: 100%|██████████| 3/3 [00:51<00:00, 17.09s/it]


In [3]:
from transformers import pipeline, AutoProcessor, AutoModelForImageTextToText
from PIL import Image


In [2]:
from llava.mm_utils import tokenizer_image_token

In [2]:
from transformers import AutoConfig, AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import torch
from llava.mm_utils import tokenizer_image_token
path = "/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/llava-1.5-7b-hf"

# 加载组件
config = AutoConfig.from_pretrained(path, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(path, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(path, config=config, trust_remote_code=True).to("cuda:0")

# 推理示例
img = Image.open("/home/NCUT/teacher/xmy/MRAG/PMI/test/url1.jpg").convert("RGB")
prompt = "Describe this image."
prompt_with_img = tokenizer_image_token(prompt, tokenizer)
inputs = processor(text=prompt_with_img, images=img, return_tensors="pt").to("cuda:0")
outputs = model.generate(**inputs, max_new_tokens=100)
print(processor.batch_decode(outputs, skip_special_tokens=True)[0])

/home/NCUT/teacher/xmy/anaconda3/envs/llava2/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/NCUT/teacher/xmy/anaconda3/envs/llava2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100%|██████████| 3/3 [00:08<00:00,  2.95s/it]


ValueError: Invalid input type. Check that `images` and/or `text` are valid inputs.

In [4]:
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoProcessor,
    AutoModelForImageTextToText
)
from PIL import Image
import torch

# === 1. 模型路径 & 设备设置 ===
path = "/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/llava-1.5-7b-hf"
device = "cuda:4"

# === 2. 加载模型组件 ===
# trust_remote_code=True 用于加载自定义模型代码
config = AutoConfig.from_pretrained(path, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(path, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    path, config=config, trust_remote_code=True
).to(device)

# === 3. 准备图像 ===
img = Image.open("/home/NCUT/teacher/xmy/MRAG/PMI/test/url1.jpg").convert("RGB")

# === 4. 构建对话历史 & 使用 apply_chat_template 格式化 ===
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image"}, 
            {"type": "text", "text": "Describe this image."}
        ]
    }
]
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
)

# === 5. 合并图像输入，并迁移到设备 ===
pixel_values = processor(images=img, return_tensors="pt")["pixel_values"]
inputs["pixel_values"] = pixel_values.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# === 6. 生成回答 ===
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200)

# === 7. 解码并打印输出 ===
answer = processor.batch_decode(outputs, skip_special_tokens=True)[0]
print("模型回答：", answer)


Loading checkpoint shards: 100%|██████████| 3/3 [00:07<00:00,  2.64s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 172.00 MiB. GPU 4 has a total capacty of 44.40 GiB of which 9.50 MiB is free. Process 2320796 has 34.87 GiB memory in use. Including non-PyTorch memory, this process has 9.50 GiB memory in use. Of the allocated memory 9.07 GiB is allocated by PyTorch, and 15.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

## 师兄给的版本

In [1]:
import sys
sys.path.append("/home/NCUT/teacher/xmy/llava-yl")  # 使用师兄的文件
from PIL import Image
import torch
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path, tokenizer_image_token, process_images
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from transformers import TextStreamer

# ========== 1. 加载模型 ==========
model_path = "/home/NCUT/teacher/xmy/llava-yl/llava-v1.5-7b"
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name=get_model_name_from_path(model_path),
    device = "cuda:4"  # 设置位置
)

# ========== 2. 设置 GPU ==========
device_id = 4
device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# ========== 3. 图像预处理 ==========
image_path = "/home/NCUT/teacher/xmy/MRAG/PMI/test/url1.jpg"
image = Image.open(image_path).convert("RGB")
images = [image]

# 统一大小处理
image_tensor = process_images(images, image_processor, model.config)
image_tensor = image_tensor.to(device=device, dtype=torch.float16)
_, _, H, W = image_tensor.shape
image_sizes = [(W, H)] * len(images)

# ========== 4. Prompt 构造 ==========
question = "Describe the scene in the image."
prompt = f"USER: {DEFAULT_IMAGE_TOKEN}\n{question} ASSISTANT:"

# 判断是否需要添加图像 start/end token（针对 mm_use_im_start_end 模型配置）
if getattr(model.config, 'mm_use_im_start_end', False):
    prompt = prompt.replace(
        DEFAULT_IMAGE_TOKEN,
        DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN
    )

# ========== 5. Tokenizer 特殊处理（注意：不是 tokenizer，而是 tokenizer_image_token）==========
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt')
input_ids = input_ids.unsqueeze(0).to(device)

# ========== 6. Streamer 推理 ==========
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# 由于使用 streamer，会自动打印结果，所以不要再尝试 decode 输出
with torch.no_grad():
    model.generate(
        input_ids=input_ids,
        images=image_tensor,
        image_sizes=image_sizes,
        do_sample=False,
        max_new_tokens=512,
        streamer=streamer
    )

# 如果你希望自己收集输出内容，请不要使用 streamer，改成：
# output_ids = model.generate(...)
# output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)


/home/NCUT/teacher/xmy/anaconda3/envs/Few_shot_llava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/home/NCUT/teacher/xmy/anaconda3/envs/Few_shot_llava/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.29s/it]


AttributeError: 'NoneType' object has no attribute 'shape'

In [1]:
import base64

def encode_image_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

params = {
    "prompt": "USER: <image>\nDescribe the image in detail. ASSISTANT:",
    "images": [encode_image_to_base64("/home/NCUT/teacher/xmy/MRAG/VLKEB-main/datasets/VLKEB_images/mmkb_images/m.0_7z2/bing_2.jpg")],  # 替换为你图片路径
    "temperature": 0.2,  # 越小越确定性
    "top_p": 1.0,
    "max_new_tokens": 512,
    "stop": None  # 或比如 "###"（触发停止解码）
}


In [2]:
import sys
sys.path.append("/home/NCUT/teacher/xmy/llava-yl")  # 使用师兄的文件
from PIL import Image
import torch
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path, tokenizer_image_token, process_images
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from transformers import TextStreamer

class llava_class:
    def __init__(self):
        model_path = "/home/NCUT/teacher/xmy/llava-yl/llava-v1.5-7b"
        self.device = "cuda:0"
        self.tokenizer, self.model, self.image_processor, self.context_len = load_pretrained_model(
            model_path=model_path,
            model_base=None,
            model_name=get_model_name_from_path(model_path),
            device = self.device,  # 设置位置
            device_map = None     # 禁止自动分布式加载
        )
        
        self.model.to(self.device)


/home/NCUT/teacher/xmy/anaconda3/envs/Few_shot_llava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
llava_model = llava_class()

You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/home/NCUT/teacher/xmy/anaconda3/envs/Few_shot_llava/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.74s/it]


In [4]:
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.mm_utils import process_images, load_image_from_base64, tokenizer_image_token
from transformers import TextIteratorStreamer

In [5]:
from PIL import Image

def resize_for_model(img, size=(336, 336)):
    return img.resize(size, Image.BICUBIC)

In [6]:


def generate_response(llava_, params):
        tokenizer, model, image_processor = llava_.tokenizer, llava_.model, llava_.image_processor
        print("model device:", next(model.parameters()).device)
        prompt = params["prompt"]
        ori_prompt = prompt
        images = params.get("images", None)
        num_image_tokens = 0
        if images is not None and len(images) > 0:
            if len(images) > 0:
                if len(images) != prompt.count(DEFAULT_IMAGE_TOKEN):
                    raise ValueError("Number of images does not match number of <image> tokens in prompt")

                images = [resize_for_model(load_image_from_base64(image),(336,336)) for image in images]
                image_sizes = [image.size for image in images]
                images = process_images(images, image_processor, model.config)

                if type(images) is list:
                    images = [image.to(llava_.device, dtype=torch.float16) for image in images]
                else:
                    images = images.to(llava_.device, dtype=torch.float16)

                replace_token = DEFAULT_IMAGE_TOKEN
                if getattr(llava_.model.config, 'mm_use_im_start_end', False):
                    replace_token = DEFAULT_IM_START_TOKEN + replace_token + DEFAULT_IM_END_TOKEN
                prompt = prompt.replace(DEFAULT_IMAGE_TOKEN, replace_token)

                num_image_tokens = prompt.count(replace_token) * model.get_vision_tower().num_patches
            else:
                images = None
                image_sizes = None
            image_args = {"images": images, "image_sizes": image_sizes}
        else:
            images = None
            image_args = {}

        temperature = float(params.get("temperature", 1.0))
        top_p = float(params.get("top_p", 1.0))
        max_context_length = getattr(model.config, 'max_position_embeddings', 2048)
        max_new_tokens = min(int(params.get("max_new_tokens", 256)), 1024)
        stop_str = params.get("stop", None)
        do_sample = True if temperature > 0.001 else False

        input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(llava_.device)
        keywords = [stop_str]
        # stopping_criteria = KeywordsStoppingCriteria(keywords, tokenizer, input_ids)
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=15)

        max_new_tokens = min(max_new_tokens, max_context_length - input_ids.shape[-1] - num_image_tokens)

        if max_new_tokens < 1:
            return json.dumps({"text": ori_prompt + "Exceeds max token length. Please start a new conversation, thanks.", "error_code": 0}).encode() + b"\0"

        print("input_ids shape:", input_ids.shape)
        print("max_new_tokens:", max_new_tokens)
        print("image tensor shape:", images.shape if images is not None else "None")
        print("image_sizes:", image_sizes)
        
        print("input_ids.device:", input_ids.device)
        print("model device:", next(model.parameters()).device)
        vision_device = next(model.get_vision_tower().parameters()).device
        print("vision tower device:", vision_device)
        model.get_vision_tower().to("cuda:2")

        if images is not None:
            if isinstance(images, list):
                for i, img in enumerate(images):
                    print(f"images[{i}].device:", img.device)
            else:
                print("images.device:", images.device)

        # 强制视觉塔迁移并触发 lazy load
        vision_tower = llava_.model.get_vision_tower()
        vision_tower.to(llava_.device)

        # dummy forward
        with torch.no_grad():
            dummy_image = torch.zeros((1, 3, 336, 336), dtype=torch.float16, device=llava_.device)
            _ = vision_tower(dummy_image)

        output_ids = model.generate(
            inputs=input_ids,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            **image_args
        )

        return output_ids
        # output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # return output_text

        # generated_text = ori_prompt
        # for new_text in streamer:
        #     generated_text += new_text
        #     if generated_text.endswith(stop_str):
        #         generated_text = generated_text[:-len(stop_str)]
        #     yield json.dumps({"text": generated_text, "error_code": 0}).encode() + b"\0"

In [7]:
!export CUDA_LAUNCH_BLOCKING=1
generate_response(llava_model,params)

model device: cuda:0
input_ids shape: torch.Size([1, 20])
max_new_tokens: 512
image tensor shape: torch.Size([1, 3, 336, 336])
image_sizes: [(336, 336)]
input_ids.device: cuda:0
model device: cuda:0
vision tower device: cuda:0
images.device: cuda:0


/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1422: indexSelectLargeIndex: block: [291,0,0], thread: [96,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1422: indexSelectLargeIndex: block: [291,0,0], thread: [97,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1422: indexSelectLargeIndex: block: [291,0,0], thread: [98,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1422: indexSelectLargeIndex: block: [291,0,0], thread: [99,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1422: indexSelectLargeIndex: block: [291,0,0], thread: [100,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1422: indexSelectLargeIndex: block: [291,0,0], thread: [101,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [8]:
print("vision patch count:", llava_model.model.get_vision_tower().num_patches)



vision patch count: 576


## 从师兄的代码调用

In [ ]:
image = load_image(image_file)
image_size = image.size

In [ ]:
image_tensor = process_images([image], image_processor, model.config)
            if type(image_tensor) is list:
                image_tensor = [image.to(model.device, dtype=torch.float16) for image in image_tensor]
            else:
                image_tensor = image_tensor.to(model.device, dtype=torch.float16)

In [ ]:
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(model.device)

In [1]:
with torch.inference_mode():
                output,image_position = model.generate(
                    input_ids,
                    images=image_tensor,
                    image_sizes=[image_size],
                    output_hidden_states=True,
                    output_attentions=True,
                    return_dict_in_generate=True,
                    max_new_tokens=args.max_new_tokens,
                    logits_processor= LogitsProcessorList([logits_processor]),
                    use_cache=True)

NameError: name 'torch' is not defined

# 使用师兄调用过的函数，模型

In [35]:
import sys  # 添加 sys 模块以便退出程序
sys.path.append("/home/NCUT/teacher/xmy/llava-yl")

In [36]:
import argparse
import torch

from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import process_images, tokenizer_image_token, get_model_name_from_path

from PIL import Image

import requests
from PIL import Image
import json
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "0"
os.environ['CUDA_VISIBLE_DEVICES'] = "3"

# from .PAI_attention import llama_modify
# from .CFG import CFGLogits

import shutil
from io import BytesIO
import plotly.io as pio
import plotly.express as px
from transformers import TextStreamer
from collections import defaultdict
from transformers.generation.logits_process import LogitsProcessorList

In [37]:
def load_image(image_file):
    if image_file.startswith('http://') or image_file.startswith('https://'):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_file).convert('RGB')
    return image

In [38]:
disable_torch_init()

In [39]:
model_path = "/home/NCUT/teacher/xmy/llava-yl/llava-v1.5-7b"

In [40]:
model_name = get_model_name_from_path(model_path)

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name = model_name,         # 通常根据 model_path 自动生成
    load_8bit=False,
    load_4bit=False,
    device="cuda"
)


You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.03s/it]


In [41]:
image_file = "/home/NCUT/teacher/xmy/MRAG/PMI/test/url1.jpg"

In [42]:
image = load_image(image_file)
image_size = image.size
print(image)
print(image_size)

<PIL.Image.Image image mode=RGB size=1500x1125 at 0x72A5880C41C0>
(1500, 1125)


In [43]:
image_tensor = process_images([image], image_processor, model.config)
if type(image_tensor) is list:
    image_tensor = [image.to(model.device, dtype=torch.float16) for image in image_tensor]
else:
    image_tensor = image_tensor.to(model.device, dtype=torch.float16)

In [44]:
conv = conv_templates["llava_v0"].copy()

In [45]:
if "mpt" in model_name.lower():
    roles = ('user', 'assistant')
else:
    roles = conv.roles

In [46]:
roles

('Human', 'Assistant')

In [47]:
inp = "describe the picture please"

In [48]:
if image is not None:
# first message
    if model.config.mm_use_im_start_end:
        inp = inp.replace(DEFAULT_IMAGE_TOKEN,DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN)
    image = None

conv.append_message(conv.roles[0], inp)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(model.device)


In [49]:
max_new_tokens = 50

In [50]:
with torch.inference_mode():
    output,image_position = model.generate(
        input_ids,
        images=image_tensor,
        image_sizes=[image_size],
        output_hidden_states=True,
        output_attentions=True,
        return_dict_in_generate=True,
        max_new_tokens=max_new_tokens,
        use_cache=True)

UnboundLocalError: local variable 'image_position' referenced before assignment